In [ ]:
# ==========================================
# CELLULE 1 : Imports et Paramétrage
# ==========================================
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
import gc
from collections import defaultdict, Counter
from itertools import combinations

warnings.filterwarnings('ignore')

PARAMS = {
    "REPEAT_WEIGHT": 1.0,
    "COOC_WEIGHT": 0.1,
    "TRANSITION_WEIGHT": 0.1,
    "HALF_LIFE_DAYS": 30,
    "STOCK_BOOST": 1.0,
    "COOC_METRIC": "conditional",
    "TOP_N_POPULAR": 50,
    "TOP_N_NEIGHBORS": 50,
    "RECENT_POP_WINDOW_DAYS": 90,
    "OUTLIER_CLIENT_ID": 784139077380372667,
    "WHOLESALE_CAP": 100,
    "MAX_CANDIDATES": 100
}
transactions = pd.read_csv("transactions.csv")
clients = pd.read_csv("clients.csv")
stocks= pd.read_csv("stocks.csv")
stores= pd.read_csv("stores.csv")
products= pd.read_csv("products.csv")
test_clients = pd.read_csv("test_clients_validation.csv")["ClientID"]


In [ ]:
# ==========================================
# CELLULE 2 : Fonctions de Data Processing
# ==========================================
def remove_invalide_clients(transactions, outlier_client_id, wholesale_cap=100):
    transactions = transactions[transactions["ClientID"] != outlier_client_id].copy()
    distinct_products_per_client = transactions.groupby("ClientID")["ProductID"].nunique()
    retail_clients = set(
        distinct_products_per_client[distinct_products_per_client <= wholesale_cap].index
    )
    transactions = transactions[transactions["ClientID"].isin(retail_clients)].copy()
    return transactions

def encode_categories(transactions, products, clients):
    transactions = transactions.copy()
    clients = clients.copy()
    products = products.copy()

    products["Marque"] = products["FamilyLevel2"].str.split().str[0].str.capitalize()

    transactions = transactions.merge(
        products[["ProductID", "Category", "Marque"]], on="ProductID", how="left"
    )

    achats_marques = transactions[["ClientID", "ProductID", "Marque"]]
    marque_preferee_df = (
        achats_marques.groupby(["ClientID", "Marque"]).size().reset_index(name="Count")
        .sort_values(["ClientID", "Count"], ascending=[True, False])
        .groupby("ClientID").first().reset_index()
        .rename(columns={"Marque": "Marque_Preferee"})
    )

    achats_categories = transactions[["ClientID", "ProductID", "Category"]]
    categorie_preferee_df = (
        achats_categories.groupby(["ClientID", "Category"]).size().reset_index(name="Count")
        .sort_values(["ClientID", "Count"], ascending=[True, False])
        .groupby("ClientID").first().reset_index()
        .rename(columns={"Category": "Category_Preferee"})
    )

    clients = clients.merge(
        marque_preferee_df[["ClientID", "Marque_Preferee"]], on="ClientID", how="left"
    ).fillna({"Marque_Preferee": "Inconnu"})
    clients = clients.merge(
        categorie_preferee_df[["ClientID", "Category_Preferee"]], on="ClientID", how="left"
    ).fillna({"Category_Preferee": "Inconnu"})

    client_columns = ["ClientGender", "ClientSegment", "ClientCountry", "Marque_Preferee", "Category_Preferee"]
    client_encoded_columns = [
        "ClientGender_enc", "ClientSegment_enc", "ClientCountry_enc",
        "Marque_Preferee_enc", "Category_Preferee_enc",
    ]
    for column, encoded_column in zip(client_columns, client_encoded_columns):
        clients[encoded_column] = clients[column].astype("category").cat.codes

    product_columns = ["Category", "Universe", "Marque"]
    product_encoded_columns = ["Category_enc", "Universe_enc", "Marque_enc"]
    for column, encoded_column in zip(product_columns, product_encoded_columns):
        products[encoded_column] = products[column].astype("category").cat.codes

    return transactions, clients, products

In [ ]:
#CANDIDATE GENERATION AND FEATURE ENGINEERING



def build_model_data(tx, clients_df, stocks_df, products_df):
    """
    Precalcule, a partir d'un historique de transactions `tx`, les structures
    necessaires au scoring des candidats :

    - listes de popularite de repli (cold-start), sur une fenetre recente
      (PARAMS['RECENT_POP_WINDOW_DAYS']), a plusieurs niveaux de granularite :
      profil demographique (pays x genre x segment) > genre seul > pays seul > global
    - matrice de co-occurrence produit-produit ponderee par recence, exprimee
      en probabilite conditionnelle P(B achete | A achete), calculee panier par panier
    - matrice de transitions sequentielles (produit achete juste apres un autre)
    - velocite produit sur les 30 derniers jours (signal de tendance)
    - compteurs d'interaction client x categorie et client x marque

    Retourne un dictionnaire `md` (model data) consomme par generer_dataset_lgbm().
    """
    client_country = dict(zip(clients_df["ClientID"], clients_df["ClientCountry_enc"]))
    client_gender = dict(zip(clients_df["ClientID"], clients_df["ClientGender_enc"]))
    stock_pos_local = set(zip(stocks_df["StoreCountry"], stocks_df["ProductID"]))
    max_date_global = tx["SaleTransactionDate"].max()

    # Profil demographique large (pays x genre x segment) -- delibérément SANS la marque/
    # categorie preferee, qui rendrait ce bucket quasi unique par client (voir note ci-dessus).
    client_profiles = (
        clients_df.set_index("ClientID")[["ClientCountry_enc", "ClientGender_enc", "ClientSegment_enc"]]
        .apply(tuple, axis=1)
        .to_dict()
    )

    # --- Popularite de repli (cold-start), calculee sur une fenetre recente ---
    recent_tx = tx[tx["SaleTransactionDate"] >= max_date_global - pd.Timedelta(days=PARAMS["RECENT_POP_WINDOW_DAYS"])]

    overall_top = list(
        recent_tx.groupby("ProductID").size().sort_values(ascending=False)
        .head(PARAMS["TOP_N_POPULAR"]).items()
    )

    recent_tx_clients = recent_tx.merge(
        clients_df[["ClientID", "ClientCountry_enc", "ClientGender_enc", "ClientSegment_enc"]],
        on="ClientID", how="left",
    )

    top_by_gender = {
        gender: list(g.groupby("ProductID").size().sort_values(ascending=False).head(PARAMS["TOP_N_POPULAR"]).items())
        for gender, g in recent_tx_clients.groupby("ClientGender_enc")
    }
    top_by_profile = {
        profile: list(g.groupby("ProductID").size().sort_values(ascending=False).head(PARAMS["TOP_N_POPULAR"]).items())
        for profile, g in recent_tx_clients.groupby(["ClientCountry_enc", "ClientGender_enc", "ClientSegment_enc"])
    }
    top_by_country = {
        country: list(g.groupby("ProductID").size().sort_values(ascending=False).head(PARAMS["TOP_N_POPULAR"]).items())
        for country, g in recent_tx_clients.groupby("ClientCountry_enc")
    }

    # Compteurs d'interaction client x categorie / marque (tout l'historique)
    tx_features = tx.merge(products_df[["ProductID", "Category_enc", "Marque_enc"]], on="ProductID", how="left")
    user_cat_interactions = tx_features.groupby(["ClientID", "Category_enc"]).size().to_dict()
    user_brand_interactions = tx_features.groupby(["ClientID", "Marque_enc"]).size().to_dict()

    #  Velocite produit sur 30 jours (signal de tendance, non pondere)
    item_velocity_30d = (
        tx[tx["SaleTransactionDate"] > max_date_global - pd.Timedelta(days=30)]
        .groupby("ProductID").size().to_dict()
    )


    # Co-occurrence produit-produit, ponderee par recence du panier
    # Regroupement par (client, date) = un vrai panier, chaque panier garde SA propre date.
    baskets = tx.groupby(["ClientID", "SaleTransactionDate"])["ProductID"].apply(set)

    item_count, co_count = Counter(), Counter()
    for (client_id, basket_date), basket in baskets.items():
        days_ago = (max_date_global - basket_date).days
        basket_weight = 0.5 ** (days_ago / PARAMS["HALF_LIFE_DAYS"])
        for product_id in basket:
            item_count[product_id] += basket_weight
        if len(basket) >= 2:
            for product_a, product_b in combinations(basket, 2):
                co_count[(product_a, product_b)] += basket_weight

    # Score de co-occurrence conditionnel P(b|a) = poids_co(a,b) / poids_total(a)

    neighbor_scores = defaultdict(dict)
    for (product_a, product_b), weight in co_count.items():
        if item_count[product_a] > 0:
            neighbor_scores[product_a][product_b] = weight / item_count[product_a]
        if item_count[product_b] > 0:
            neighbor_scores[product_b][product_a] = weight / item_count[product_b]

    top_neighbors = {
        item: sorted(neighbors.items(), key=lambda x: -x[1])[: PARAMS["TOP_N_NEIGHBORS"]]
        for item, neighbors in neighbor_scores.items()
    }

    # Transitions sequentielles : produit achete juste apres un autre
    #regroupement de l'historique sous la forme clientID: [(productID, date), ...]
    client_hist_dated = tx.groupby("ClientID").apply(
                lambda g: list(zip(g["ProductID"], g["SaleTransactionDate"]))
            ).to_dict()

    trans_count = Counter()
    item_trans_base = Counter()
    for client_id, history in client_hist_dated.items():
        history_sorted = sorted(history, key=lambda x: x[1])
        for i in range(len(history_sorted) - 1):
            product_a, product_b = history_sorted[i][0], history_sorted[i + 1][0]
            if product_a != product_b:
                trans_count[(product_a, product_b)] += 1
                item_trans_base[product_a] += 1

    transition_scores = defaultdict(dict)
    for (product_a, product_b), count in trans_count.items():
        transition_scores[product_a][product_b] = count / item_trans_base[product_a]

    top_transitions = {
        item: sorted(neighbors.items(), key=lambda x: -x[1])[: PARAMS["TOP_N_NEIGHBORS"]]
        for item, neighbors in transition_scores.items()
    }

    return {
        "client_hist_dated": client_hist_dated,
        "top_neighbors": top_neighbors,
        "top_transitions": top_transitions,
        "top_by_profile": top_by_profile,
        "top_by_gender": top_by_gender,
        "top_by_country": top_by_country,
        "overall_top": overall_top,
        "client_profiles": client_profiles,
        "client_country": client_country,
        "client_gender": client_gender,
        "stock_pos": stock_pos_local,
        "max_date_global": max_date_global,
        "user_cat_int": user_cat_interactions,
        "user_brand_int": user_brand_interactions,
        "item_vel": item_velocity_30d,
    }

def generer_dataset_lgbm(liste_clients, md, prod_info, clients_df, products_df,
                          vrais_achats=None, target_date=None, batch_size=5000):
    """
    Genere, pour chaque client de `liste_clients`, l'ensemble des produits candidats
    (deja achetes, co-achetes, en transition, populaires en repli) et leurs features.
    On procède comme suit: à l'aide du dictionnaire md construit, on choisit d'abord pour chaque produit
    de l'historique du client TOP_N_NEIGHBORS voisins les plus probables,en se basant sur le score de co-occurence pondéré par la récence,,
    puis on calcule le score heuristique de tous les produits rassemblés, en pondérant toujours par le score de récence, et on choisit
    les TOP_N_NEIGHBORS entre eux, puis on ajoute les TOP_N_POPULARS produits les plus populaires en repli (cold-start) pour compléter la liste des candidats.
    """
    dfs_totaux = []
    for i in range(0, len(liste_clients), batch_size):
        batch_clients = liste_clients[i:i + batch_size]
        lignes = []

        for cid in batch_clients:
            hist = md["client_hist_dated"].get(cid, [])
            profil_tuple = md["client_profiles"].get(cid)
            country = md["client_country"].get(cid)
            gender = md["client_gender"].get(cid)

            # Cascade de repli : profil complet -> genre -> pays -> popularite globale.
            if profil_tuple in md["top_by_profile"]:
                fallback = md["top_by_profile"][profil_tuple]
            elif gender in md["top_by_gender"]:
                fallback = md["top_by_gender"][gender]
            elif country in md["top_by_country"]:
                fallback = md["top_by_country"][country]
            else:
                fallback = md["overall_top"]

            # Isolement du dernier produit achete (pour Score_Trans_Last).
            ref_date = target_date or (max(d for _, d in hist) if hist else md["max_date_global"])
            hist_sorted = sorted([h for h in hist if (ref_date - h[1]).days >= 0], key=lambda x: x[1])
            last_item = hist_sorted[-1][0] if hist_sorted else None

            # Poids de recence de chaque produit deja achete par le client.
            product_weight = {}
            for pid, d in hist:
                days_ago = (ref_date - d).days
                if days_ago < 0:
                    continue
                w = 0.5 ** (days_ago / PARAMS["HALF_LIFE_DAYS"])
                product_weight[pid] = product_weight.get(pid, 0) + w

            score_recence, score_cooc, score_trans_hist, score_trans_last = {}, {}, {}, {}

            # Score_Trans_Last : transition depuis le dernier produit achete uniquement,
            # sans ponderation £
            if last_item:
                for neigh, s in md["top_transitions"].get(last_item, []):
                    score_trans_last[neigh] = s

            for pid, w in product_weight.items():
                score_recence[pid] = score_recence.get(pid, 0) + PARAMS["REPEAT_WEIGHT"] * w
                for neigh, s in md["top_neighbors"].get(pid, []):
                    score_cooc[neigh] = score_cooc.get(neigh, 0) + PARAMS["COOC_WEIGHT"] * w * s
                for neigh, s in md["top_transitions"].get(pid, []):
                    score_trans_hist[neigh] = score_trans_hist.get(neigh, 0) + PARAMS["TRANSITION_WEIGHT"] * w * s

            candidats = set(score_recence) | set(score_cooc) | set(score_trans_hist) | set(score_trans_last)
            candidats |= set(p for p, _ in fallback[: PARAMS["TOP_N_POPULAR"]])

            client_lignes = []
            for pid in candidats:
                sr = score_recence.get(pid, 0.0)
                sc = score_cooc.get(pid, 0.0)
                sth = score_trans_hist.get(pid, 0.0)
                stl = score_trans_last.get(pid, 0.0)

                en_stock = 1 if (country, pid) in md["stock_pos"] else 0
                sh = sr + sc + sth + stl
                if en_stock:
                    sh *= PARAMS["STOCK_BOOST"]

                c_cat = prod_info.get(pid, {}).get("Category_enc", -1)
                c_brand = prod_info.get(pid, {}).get("Marque_enc", -1)

                ligne = {
                    "ClientID": cid, "ProductID": pid,
                    "Score_Recence_Achat": np.float32(sr),
                    "Score_Cooc_Conditional": np.float32(sc),
                    "Score_Trans_Hist": np.float32(sth),
                    "Score_Trans_Last": np.float32(stl),
                    "En_Stock_Local": np.int8(en_stock),
                    "Score_Heuristique_Pur": np.float32(sh),
                    "User_Cat_Buy_Count": np.float32(md["user_cat_int"].get((cid, c_cat), 0)),
                    "User_Brand_Buy_Count": np.float32(md["user_brand_int"].get((cid, c_brand), 0)),
                    "Item_Velocity_30d": np.float32(md["item_vel"].get(pid, 0)),
                    "Is_Repurchase": np.int8(1 if sr > 0 else 0),
                }
                if vrais_achats is not None:
                    ligne["Label"] = np.int8(1) if (cid, pid) in vrais_achats else np.int8(0)
                client_lignes.append(ligne)

            # Plafonne le nombre de candidats par client au score heuristique le plus eleve.
            if len(client_lignes) > PARAMS["MAX_CANDIDATES"]:
                client_lignes = sorted(client_lignes, key=lambda x: x["Score_Heuristique_Pur"], reverse=True)[: PARAMS["MAX_CANDIDATES"]]

            lignes.extend(client_lignes)

        df_batch = pd.DataFrame(lignes)
        if not df_batch.empty:
            cols_client = ["ClientID", "ClientGender_enc", "ClientSegment_enc", "ClientCountry_enc",
                            "Marque_Preferee_enc", "Category_Preferee_enc"]
            cols_produit = ["ProductID", "Category_enc", "Universe_enc", "Marque_enc"]
            df_batch = df_batch.merge(clients_df[cols_client], on="ClientID", how="left")
            df_batch = df_batch.merge(products_df[cols_produit], on="ProductID", how="left")
            df_batch["Est_Marque_Preferee"] = (df_batch["Marque_enc"] == df_batch["Marque_Preferee_enc"]).astype(np.int8)
            # Feature analogue pour la categorie preferee (Category_Preferee_enc etait calculee
            # mais jamais utilisee comme feature -- ajoutee ici par symetrie avec la marque).
            df_batch["Est_Categorie_Preferee"] = (df_batch["Category_enc"] == df_batch["Category_Preferee_enc"]).astype(np.int8)
            dfs_totaux.append(df_batch)
        gc.collect()

    return pd.concat(dfs_totaux, ignore_index=True) if dfs_totaux else pd.DataFrame()

In [ ]:

# ==========================================
# CELLULE 4 : Exécution Principale (Data Loading & Prep)
# ==========================================

transactions = pd.read_csv("transactions.csv", parse_dates=["SaleTransactionDate"])
clients = pd.read_csv("clients.csv")
products = pd.read_csv("products.csv")
stocks = pd.read_csv("stocks.csv")
test_clients = pd.read_csv("test_clients_validation.csv")["ClientID"].unique()

transactions = remove_invalide_clients(
    transactions, PARAMS["OUTLIER_CLIENT_ID"], PARAMS["WHOLESALE_CAP"]
)
transactions, clients, products = encode_categories(transactions, products, clients)

prod_info = products.set_index("ProductID")[["Category_enc", "Marque_enc"]].to_dict("index")

print(f"Transactions apres nettoyage : {len(transactions)}")
print(f"Clients : {clients['ClientID'].nunique()}")
print(f"Clients de validation : {len(test_clients)}")




Transactions apres nettoyage : 1090734
Clients : 424037
Clients de validation : 300
Nombre de clients pour l'entrainement : 72847
Dataset d'entrainement : (7265474, 23)
Dataset de test : (29931, 22)
Debut de l'entrainement...
Fichier genere : submission_hybrid_028.csv


## ENTRAINEMENT ET PREDICTION

In [ ]:
import lightlbm as lgb

## SPLIT ONE OUT
Pour chaque client, on isole sa toute dernière visite (verité terrain à prédire) du reste de
son historique (utilisé pour calculer les features). Seuls les clients ayant au moins 3 lignes
d'historique avant cette dernière visite sont conservés, pour disposer d'un minimum de signal.

In [ ]:



last_visit_dates = transactions.groupby("ClientID")["SaleTransactionDate"].max().reset_index()
last_visit_dates.rename(columns={"SaleTransactionDate": "LastVisitDate"}, inplace=True)
tx_with_last = transactions.merge(last_visit_dates, on="ClientID")

train_history_full = tx_with_last[tx_with_last["SaleTransactionDate"] < tx_with_last["LastVisitDate"]].copy()
train_target_full = tx_with_last[tx_with_last["SaleTransactionDate"] == tx_with_last["LastVisitDate"]].copy()

client_counts = train_history_full.groupby("ClientID").size()
valid_clients = client_counts[client_counts >= 3].index

train_history = train_history_full[train_history_full["ClientID"].isin(valid_clients)].copy()
train_target = train_target_full[train_target_full["ClientID"].isin(valid_clients)].copy()
print(f"Nombre de clients pour l'entrainement : {len(valid_clients)}")

##  construction des jeux d'entraînement et de test

`md_train` est calculé sur `train_history` uniquement (pas de fuite : la visite cachée
n'intervient qu'au moment de labelliser les candidats via `vrais_achats`).
`md_test` est recalculé sur `transactions` en entier, puisqu'il s'agit ici de vraies
prédictions futures pour les clients de validation, sans rien à cacher.

In [ ]:
md_train = build_model_data(train_history, clients, stocks, products)
vrais_achats = set(zip(train_target["ClientID"], train_target["ProductID"]))
df_train = generer_dataset_lgbm(
    train_target["ClientID"].unique(), md_train, prod_info, clients, products,
    vrais_achats=vrais_achats, target_date=None,
)

md_test = build_model_data(transactions, clients, stocks, products)
df_test = generer_dataset_lgbm(test_clients, md_test, prod_info, clients, products)

print(f"Dataset d'entrainement : {df_train.shape}")
print(f"Dataset de test : {df_test.shape}")

features = [
    "Score_Recence_Achat", "Score_Cooc_Conditional",
    "Score_Trans_Hist", "Score_Trans_Last",
    "En_Stock_Local", "Score_Heuristique_Pur",
    "User_Cat_Buy_Count", "User_Brand_Buy_Count", "Item_Velocity_30d", "Is_Repurchase",
    "ClientGender_enc", "ClientSegment_enc", "ClientCountry_enc",
    "Category_enc", "Universe_enc", "Marque_enc",
    "Est_Marque_Preferee", "Est_Categorie_Preferee",
]
cat_features = [
    "ClientGender_enc", "ClientSegment_enc", "ClientCountry_enc",
    "Category_enc", "Universe_enc", "Marque_enc",
]

# Tri par ClientID obligatoire pour l'API de ranking LightGBM : chaque groupe (grp)
# represente l'ensemble des candidats d'un meme client.
df_train = df_train.sort_values("ClientID", kind="stable").reset_index(drop=True)
grp = df_train.groupby("ClientID", sort=False).size().values

train_data = lgb.Dataset(
    df_train[features],
    label=df_train["Label"],
    group=grp,
    categorical_feature=cat_features,
    free_raw_data=False,
)

params = {
    "objective": "lambdarank",
    "metric": "map",
    "eval_at": [5],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambdarank_truncation_level": 10,
    "random_state": 42,
    "verbose": -1,
}

model = lgb.train(params, train_data, num_boost_round=250)

##  prédiction et stratégie de repli

Score de chaque candidat par le modèle, tri décroissant, top 5 conservé par client. Si un
client a moins de 5 candidats scorés (historique trop pauvre), la liste est complétée avec la
popularité de repli démographique (profil → pays → global), sans doublon.

In [ ]:
df_test = df_test.sort_values("ClientID", kind="stable").reset_index(drop=True)
df_test["Score_LGBM_Brut"] = model.predict(df_test[features])

df_trie = df_test.sort_values(by=["ClientID", "Score_LGBM_Brut"], ascending=[True, False])

preds = []
for client, group in df_trie.groupby("ClientID"):
    result = list(group["ProductID"].values[:5])

    profil_tuple = md_test["client_profiles"].get(client)
    country = md_test["client_country"].get(client)

    if profil_tuple in md_test["top_by_profile"]:
        fallback_list = md_test["top_by_profile"][profil_tuple]
    elif country in md_test["top_by_country"]:
        fallback_list = md_test["top_by_country"][country]
    else:
        fallback_list = md_test["overall_top"]

    idx = 0
    while len(result) < 5 and idx < len(fallback_list):
        pid_fallback = fallback_list[idx][0]
        if pid_fallback not in result:
            result.append(pid_fallback)
        idx += 1

    preds.append({
        "ClientID": client, "item_1": result[0], "item_2": result[1],
        "item_3": result[2], "item_4": result[3], "item_5": result[4],
    })

pd.DataFrame(preds).assign(KEY="Validation").to_csv("submission_finale.csv", index=False)
print("Fichier genere : submission_finale")